# 13 - Overlay: theme conviction vs ETF price (AUTO, weekly)

Auto-picks the highest-sentiment themes and overlays each theme's conviction_z
against its ETF's price. `FREQ='W'` averages conviction weekly for a smoother
read.

In [ ]:
import os, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
P = os.path.join(ROOT, 'data', 'processed')
PRICES_PATH = os.path.join(ROOT, 'data', 'prices', 'prices.parquet')

# window comes from update_data.py (same toggle as live vs backtest)
from update_data import START_DATE, END_DATE
WIN_LO = pd.to_datetime(START_DATE)
WIN_HI = pd.to_datetime(END_DATE) if END_DATE else None
print('window:', WIN_LO.date(), '->', (WIN_HI.date() if WIN_HI is not None else 'LIVE (newest)'))

def clip_series(s):
    s = s[s.index >= WIN_LO]
    return s if WIN_HI is None else s[s.index <= WIN_HI]

def clip_dates(df, col):
    df = df[df[col] >= WIN_LO]
    return df if WIN_HI is None else df[df[col] <= WIN_HI]

def load_prices():
    if not os.path.exists(PRICES_PATH):
        raise FileNotFoundError('prices.parquet not found - run  python pull_bloomberg_prices.py  first.')
    px = pd.read_parquet(PRICES_PATH); px['date'] = pd.to_datetime(px['date'])
    return px

def price_series(prices, symbol):
    # daily close, then made CONTINUOUS (forward-fill weekends/holidays) so the
    # line is smooth with no gaps. Clip to the window.
    one = prices[prices['symbol'] == symbol].sort_values('date')
    s = one.set_index('date')['px_last']
    if not s.empty:
        s = s.asfreq('D').ffill()
    return clip_series(s)


In [ ]:
from src.themes import THEME_ETFS
HOW_MANY = 6
FREQ = 'W'        # 'W' weekly, 'D' daily, 'M' monthly
MIN_POSTS = 30    # ignore barely-discussed themes when ranking sentiment

In [ ]:
tsent = pd.read_parquet(os.path.join(P, 'daily_theme_sentiment.parquet'))
tsent['date'] = pd.to_datetime(tsent['date']); tsent = clip_dates(tsent, 'date')
posts = tsent.groupby('theme')['n_posts'].sum(); mood = tsent.groupby('theme')['net_bullish'].mean()
themes = mood[posts >= MIN_POSTS].sort_values(ascending=False).head(HOW_MANY).index.tolist()
print('auto themes (highest sentiment):', themes)
conv = pd.read_parquet(os.path.join(P, 'daily_theme_conviction.parquet'))
conv['date'] = pd.to_datetime(conv['date'])
prices = load_prices()
for theme in themes:
    etf = THEME_ETFS.get(theme)
    if etf is None: print('skip', theme, '- no ETF'); continue
    c = clip_series(conv[conv['theme'] == theme].sort_values('date').set_index('date')['conviction_z'])
    px = price_series(prices, etf)
    if px.empty: print('skip', theme, f'- no price for {etf}'); continue
    if FREQ != 'D':
        c = c.resample(FREQ).mean(); px = px.resample(FREQ).last()
    fig, ax1 = plt.subplots(figsize=(13, 5))
    ax1.axhline(0, color='black', linewidth=0.6)
    ax1.plot(c.index, c.values, color='tab:purple', linewidth=1.8, label='conviction_z')
    ax1.set_ylabel('conviction z', color='tab:purple'); ax1.tick_params(axis='y', labelcolor='tab:purple')
    ax2 = ax1.twinx(); ax2.plot(px.index, px.values, color='tab:red', linewidth=1.8, label='price')
    ax2.set_ylabel('price (USD)', color='tab:red'); ax2.tick_params(axis='y', labelcolor='tab:red')
    ax1.set_title(f'{theme}: conviction vs {etf} price ({FREQ})'); ax1.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()